# Notebook 2 — A2UI with AgentCore

**Agentic AI Practitioner Bootcamp · Final Day**

Notebook 1 built the A2UI pattern on a Strands agent. This notebook puts the **same** TravelMind agent into the AWS production frame: AgentCore Runtime to host it, Identity to drive role-based views, Policy to enforce the approval gate server-side, and Observability to feed the admin view.

It is standalone. The compact A2UI library is re-included so you can run this without Notebook 1.

### What each AgentCore service maps to

| AgentCore service | Status | Role in the UI layer | In this notebook |
|---|---|---|---|
| Runtime | GA (Oct 2025) | Hosts the agent, A2A transport, 8-hour sessions, isolation | Real SDK, run locally; deploy commands shown |
| Identity | GA | Custom claims drive role-based views | Simulated claims, real role filtering logic |
| Policy | Preview (Dec 2025, Cedar) | Intercepts every tool call, enforces gates server-side | Cedar-style policy + local evaluator |
| Observability | GA | CloudWatch + OpenTelemetry, trajectory inspection | Local trace analog; enable command shown |

What is **real**: the Runtime SDK entrypoint, the A2UI payloads, and all the role and policy logic. What is **simulated**: the issued Identity claims, the Policy interception, and the cloud trace sink. Each is labelled where it appears.

## 0. Setup and config

### Install (run once)

```bash
python -m venv .venv && source .venv/bin/activate
pip install strands-agents bedrock-agentcore bedrock-agentcore-starter-toolkit boto3
```

| Setting | Value | Note |
|---|---|---|
| Region | `us-east-1` | TravelMind anchor. The starter toolkit defaults to `us-west-2`; set region explicitly |
| Model | `us.anthropic.claude-haiku-4-5-20251001-v1:0` | Inference profile, same as Notebook 1 |
| Runtime entrypoint | `@app.entrypoint` on `BedrockAgentCoreApp` | Wraps your existing agent, any framework |
| Deploy needs | Docker or Finch + an IAM execution role | Not required to run this notebook |

In [1]:
# Run the pip line once in your venv (commented so this notebook never reinstalls).
# !pip install strands-agents bedrock-agentcore bedrock-agentcore-starter-toolkit boto3

import json
from datetime import datetime, timezone

CONFIG = {
    "aws_region": "us-east-1",
    "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "pnr": "JX48Q2",
}


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


print("config:", json.dumps(CONFIG, indent=2))

config: {
  "aws_region": "us-east-1",
  "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
  "pnr": "JX48Q2"
}


## 1. The A2UI library (compact, standalone)

Same v0.8 builders, validator, and tiny renderer from Notebook 1, condensed into one cell so this notebook stands alone.

In [2]:
KNOWN_TYPES = {"Column", "Row", "Text", "Button", "MetricCard"}
CONTAINER = {"Column", "Row"}
TOP_KEYS = {"surfaceUpdate", "dataModelUpdate", "beginRendering", "deleteSurface", "userAction"}


def text(cid, value=None, path=None):
    b = {}
    if path is not None: b["path"] = path
    if value is not None: b["literalString"] = value
    return {"id": cid, "component": {"Text": {"text": b}}}


def button(cid, child_id, action_name, context=None):
    a = {"name": action_name}
    if context: a["context"] = context
    return {"id": cid, "component": {"Button": {"child": child_id, "action": a}}}


def column(cid, ids): return {"id": cid, "component": {"Column": {"children": {"explicitList": list(ids)}}}}
def row(cid, ids): return {"id": cid, "component": {"Row": {"children": {"explicitList": list(ids)}}}}
def metric_card(cid, label, value): return {"id": cid, "component": {"MetricCard": {"label": {"literalString": label}, "value": {"literalString": value}}}}
def surface_update(sid, comps): return {"surfaceUpdate": {"surfaceId": sid, "components": comps}}
def begin_rendering(sid, root): return {"beginRendering": {"surfaceId": sid, "root": root}}
def user_action(name, sid, src, context=None): return {"userAction": {"name": name, "surfaceId": sid, "sourceComponentId": src, "timestamp": now_iso(), "context": context or {}}}


def validate(messages):
    errors, ids, roots, refs = [], set(), [], []
    for i, m in enumerate(messages):
        ks = [k for k in m if k in TOP_KEYS]
        if len(ks) != 1:
            errors.append(f"msg[{i}] needs one A2UI key, found {list(m.keys())}"); continue
        if ks[0] == "surfaceUpdate":
            for c in m["surfaceUpdate"].get("components", []):
                if "id" not in c or "component" not in c:
                    errors.append(f"msg[{i}] bad component {c}"); continue
                ids.add(c["id"]); tks = list(c["component"])
                if len(tks) != 1:
                    errors.append(f"'{c['id']}' must wrap one type"); continue
                t, body = tks[0], c["component"][tks[0]]
                if t not in KNOWN_TYPES: errors.append(f"'{c['id']}' unknown type '{t}'")
                if t in CONTAINER:
                    refs += [(c["id"], ch) for ch in body.get("children", {}).get("explicitList", [])]
                if t == "Button" and "child" in body: refs.append((c["id"], body["child"]))
        elif ks[0] == "beginRendering":
            roots.append(m["beginRendering"].get("root"))
    for p, r in refs:
        if r not in ids: errors.append(f"'{p}' references missing child '{r}'")
    for r in roots:
        if r not in ids: errors.append(f"root '{r}' not defined")
    return errors


def render_ascii(messages):
    comps, root = {}, None
    for m in messages:
        if "surfaceUpdate" in m:
            for c in m["surfaceUpdate"]["components"]: comps[c["id"]] = c["component"]
        elif "beginRendering" in m: root = m["beginRendering"]["root"]

    def rv(b): return b.get("literalString", b.get("path", "")) if isinstance(b, dict) else str(b)
    out = []

    def walk(cid, d):
        n = comps.get(cid)
        if not n: return
        t = list(n)[0]; body = n[t]; pad = "  " * d
        if t == "Text": out.append(pad + rv(body.get("text", {})))
        elif t == "Button":
            ch = body.get("child")
            lbl = rv(comps.get(ch, {}).get("Text", {}).get("text", {})) if ch in comps else ch
            out.append(pad + f"[ {lbl} ]  -> action:{body['action']['name']}")
        elif t == "MetricCard": out.append(pad + f"+-- {rv(body['label'])}: {rv(body['value'])} --+")
        elif t in CONTAINER:
            out.append(pad + t + ":")
            for c in body.get("children", {}).get("explicitList", []): walk(c, d + 1)

    walk(root, 0) if root else out.append("(no root)")
    s = "\n".join(out); print(s); return s


print("A2UI lib ready. self-test:")
demo = [surface_update("d", [column("root", ["t"]), text("t", "A2UI standalone OK")]), begin_rendering("d", "root")]
print("validate:", validate(demo)); render_ascii(demo)

A2UI lib ready. self-test:
validate: []
Column:
  A2UI standalone OK


'Column:\n  A2UI standalone OK'

In [3]:
# TravelMind surface builders (condensed from Notebook 1)
FLIGHTS = [
    {"id": "JX490", "depart": "18:05", "fare": "9,400"},
    {"id": "JX511", "depart": "13:40", "fare": "11,800"},
    {"id": "JX482", "depart": "09:15", "fare": "14,200"},
]


def options_surface(sid, flights, with_override=False):
    comps, rows = [], []
    for f in flights:
        ids = [f"flt-{f['id']}", f"fare-{f['id']}", f"lbl-{f['id']}", f"btn-{f['id']}"]
        comps += [text(ids[0], f"{f['id']}  dep {f['depart']}"), text(ids[1], f"INR {f['fare']}"),
                  text(ids[2], "Rebook"), button(ids[3], ids[2], "select_flight", [f"/flights/{f['id']}"])]
        rowkids = [ids[0], ids[1], ids[3]]
        if with_override:
            comps += [text(f"ovl-{f['id']}", "Override fare"), button(f"ovb-{f['id']}", f"ovl-{f['id']}", "override_fare", [f"/flights/{f['id']}"])]
            rowkids.append(f"ovb-{f['id']}")
        comps.append(row(f"row-{f['id']}", rowkids)); rows.append(f"row-{f['id']}")
    comps += [text("hdr", "Available flights"), column("root", ["hdr"] + rows)]
    return [surface_update(sid, comps), begin_rendering(sid, "root")]


def approval_surface(sid, f):
    comps = [text("q", f"Rebook {f['id']} for INR {f['fare']}?"),
             text("tool", f"tool: book_flight   flight={f['id']}   pnr={CONFIG['pnr']}"),
             text("ok-l", "Approve"), button("ok", "ok-l", "approve_booking", [f"/pending/{f['id']}"]),
             text("no-l", "Reject"), button("no", "no-l", "reject_booking"),
             column("root", ["q", "tool", "ok", "no"])]
    return [surface_update(sid, comps), begin_rendering(sid, "root")]


def confirmation_surface(sid, f):
    return [surface_update(sid, [metric_card("m", "Booking confirmed", f"{f['id']} . INR {f['fare']}"),
                                 text("p", f"PNR {CONFIG['pnr']} updated"), column("root", ["m", "p"])]),
            begin_rendering(sid, "root")]


print("builders ready for", len(FLIGHTS), "flights")

builders ready for 3 flights


## 2. AgentCore Runtime entrypoint

AgentCore Runtime hosts any-framework agents as a serverless API with session isolation and 8-hour sessions. You wrap your existing agent in `BedrockAgentCoreApp` and mark one function with `@app.entrypoint`.

We call the entrypoint **directly** here to demonstrate, which needs no server and no AWS. In production `app.run()` serves it on port 8080 and the starter toolkit deploys it.

In [4]:
from bedrock_agentcore import BedrockAgentCoreApp

app = BedrockAgentCoreApp()


@app.entrypoint
def invoke(payload):
    """TravelMind entrypoint. payload: {prompt, role}. Returns a result + an A2UI surface."""
    prompt = payload.get("prompt", "")
    role = payload.get("role", "customer")
    msgs = options_surface("flights", FLIGHTS)
    if validate(msgs):
        return {"result": "Falling back to text: JX490, JX511, JX482", "surface": None}
    return {"result": f"Showing flight options for role={role}", "surface": msgs, "role": role}


# direct call, no server, no AWS
out = invoke({"prompt": "Show me flights from Bangalore to Mumbai", "role": "customer"})
print("entrypoint result:", out["result"])
print("\nsurface the runtime would stream back:\n")
render_ascii(out["surface"])

entrypoint result: Showing flight options for role=customer

surface the runtime would stream back:

Column:
  Available flights
  Row:
    JX490  dep 18:05
    INR 9,400
    [ Rebook ]  -> action:select_flight
  Row:
    JX511  dep 13:40
    INR 11,800
    [ Rebook ]  -> action:select_flight
  Row:
    JX482  dep 09:15
    INR 14,200
    [ Rebook ]  -> action:select_flight


'Column:\n  Available flights\n  Row:\n    JX490  dep 18:05\n    INR 9,400\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX511  dep 13:40\n    INR 11,800\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX482  dep 09:15\n    INR 14,200\n    [ Rebook ]  -> action:select_flight'

**Deploying for real** (reference, not run here, needs Docker or Finch and an IAM role):

```bash
# put the @app.entrypoint code in travelmind_agent.py with app.run() under __main__
agentcore configure -e travelmind_agent.py --name travelmind
agentcore launch                 # builds, pushes, and hosts on AgentCore Runtime
agentcore invoke '{"prompt": "show flights", "role": "customer"}'
```

Local smoke test before deploy:

```bash
python travelmind_agent.py       # serves on :8080
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -d '{"prompt": "show flights", "role": "customer"}'
```

## 3. Identity → role-based views

AgentCore Identity verifies the caller and carries **custom claims** (for example a `role`). The agent reads the role and offers a different surface to each audience. The catalog and the actions are chosen by role, on the server.

Below the claims are **simulated** (in production AgentCore Identity issues and verifies them). The filtering logic is the real part.

In [5]:
# simulated decoded claims (production: issued + verified by AgentCore Identity)
CLAIMS = {
    "customer": {"sub": "u-1001", "role": "customer"},
    "agent":    {"sub": "u-2002", "role": "agent"},
    "admin":    {"sub": "u-3003", "role": "admin"},
}

# what each role is even allowed to be offered
ROLE_CATALOG = {
    "customer": {"actions": {"select_flight", "approve_booking", "reject_booking"}, "override": False},
    "agent":    {"actions": {"select_flight", "approve_booking", "reject_booking", "override_fare"}, "override": True},
    "admin":    {"actions": {"view_audit"}, "override": False},
}


def surface_for_role(role):
    if role == "admin":
        # admin sees the record layer, not the booking UI
        return [surface_update("admin", [
            metric_card("m1", "Surfaces emitted", "see audit"),
            text("note", "Admin view renders traces and audit, not booking controls"),
            column("root", ["m1", "note"])]), begin_rendering("admin", "root")]
    return options_surface("flights", FLIGHTS, with_override=ROLE_CATALOG[role]["override"])


for role in ("customer", "agent", "admin"):
    print(f"==== role: {role}  (claims sub={CLAIMS[role]['sub']}) ====")
    render_ascii(surface_for_role(role))
    print()

==== role: customer  (claims sub=u-1001) ====
Column:
  Available flights
  Row:
    JX490  dep 18:05
    INR 9,400
    [ Rebook ]  -> action:select_flight
  Row:
    JX511  dep 13:40
    INR 11,800
    [ Rebook ]  -> action:select_flight
  Row:
    JX482  dep 09:15
    INR 14,200
    [ Rebook ]  -> action:select_flight

==== role: agent  (claims sub=u-2002) ====
Column:
  Available flights
  Row:
    JX490  dep 18:05
    INR 9,400
    [ Rebook ]  -> action:select_flight
    [ Override fare ]  -> action:override_fare
  Row:
    JX511  dep 13:40
    INR 11,800
    [ Rebook ]  -> action:select_flight
    [ Override fare ]  -> action:override_fare
  Row:
    JX482  dep 09:15
    INR 14,200
    [ Rebook ]  -> action:select_flight
    [ Override fare ]  -> action:override_fare

==== role: admin  (claims sub=u-3003) ====
Column:
  +-- Surfaces emitted: see audit --+
  Admin view renders traces and audit, not booking controls



The customer sees Rebook only. The agent also gets Override fare. The admin gets the record view. Same agent, three surfaces, decided by a verified claim, not by hiding buttons on the client.

## 4. Policy → the approval gate, server-side

AgentCore Policy (preview) intercepts every tool call and evaluates **Cedar** rules before the tool runs. This is the server-side enforcement of the gate from Notebook 1.

Below is a Cedar-style policy and a local evaluator that mirrors it. In production the Policy service intercepts the call; here we call the evaluator before `book_flight`.

In [6]:
# Cedar-style policy (illustrative). AgentCore Policy uses the Cedar language.
CEDAR_POLICY = '''
permit (
    principal,
    action == Action::"book_flight",
    resource
) when {
    principal.role == "agent"
    || (principal.role == "customer" && resource.amount <= 50000)
};

forbid (
    principal,
    action == Action::"book_flight",
    resource
) when {
    principal.role == "admin"
};
'''


def evaluate_policy(principal, action, resource):
    """Local stand-in for AgentCore Policy. Returns (decision, reason)."""
    role = principal.get("role")
    if action == "book_flight":
        if role == "admin":
            return "Deny", "admin is read-only by policy"
        if role == "agent":
            return "Allow", "agent permitted"
        if role == "customer" and resource.get("amount", 0) <= 50000:
            return "Allow", "customer within ceiling"
        return "Deny", "customer over ceiling or role not permitted"
    return "Allow", "no policy gate for this action"


print(CEDAR_POLICY)
chosen = FLIGHTS[1]
amount = int(chosen["fare"].replace(",", ""))
for role in ("customer", "agent", "admin"):
    decision, why = evaluate_policy(CLAIMS[role], "book_flight", {"amount": amount})
    print(f"  role={role:9s} amount={amount}  ->  {decision}  ({why})")


permit (
    principal,
    action == Action::"book_flight",
    resource
) when {
    principal.role == "agent"
    || (principal.role == "customer" && resource.amount <= 50000)
};

forbid (
    principal,
    action == Action::"book_flight",
    resource
) when {
    principal.role == "admin"
};

  role=customer  amount=11800  ->  Allow  (customer within ceiling)
  role=agent     amount=11800  ->  Allow  (agent permitted)
  role=admin     amount=11800  ->  Deny  (admin is read-only by policy)


## 5. Observability → the admin view

AgentCore Observability emits CloudWatch metrics and OpenTelemetry traces, with step-by-step trajectory inspection. The audit log from Notebook 1 is the local analog: one span per step in the flow.

Enable the real thing once per account:

```bash
# enable CloudWatch Transaction Search, then view traces in the AgentCore console
aws logs put-account-policy ...   # see: Enabling AgentCore runtime observability
```

In [7]:
TRACE = []


def span(name, **attrs):
    TRACE.append({"ts": now_iso(), "span": name, **attrs})


# build a trajectory analogous to what Observability records
TRACE.clear()
span("invoke", role="customer", prompt="rebook me on JX511")
span("ui_emitted", surface="flights", components=len(options_surface("flights", FLIGHTS)[0]["surfaceUpdate"]["components"]))
span("user_action", action="select_flight", flight=chosen["id"])
span("ui_emitted", surface="approve", components=len(approval_surface("approve", chosen)[0]["surfaceUpdate"]["components"]))
ua = user_action("approve_booking", "approve", "ok", {"flight": chosen["id"]})
span("user_action", action=ua["userAction"]["name"], context=ua["userAction"]["context"])
decision, why = evaluate_policy(CLAIMS["customer"], "book_flight", {"amount": amount})
span("policy_decision", action="book_flight", decision=decision, reason=why)
if decision == "Allow":
    span("tool_executed", tool="book_flight", flight=chosen["id"])
    span("ui_emitted", surface="confirm", components=len(confirmation_surface("confirm", chosen)[0]["surfaceUpdate"]["components"]))


def print_trace(rows):
    for i, r in enumerate(rows):
        attrs = {k: v for k, v in r.items() if k not in ("ts", "span")}
        print(f"  {i:>2}. {r['span']:<16} {json.dumps(attrs)}")


print("trajectory (one span per step):")
print_trace(TRACE)

trajectory (one span per step):
   0. invoke           {"role": "customer", "prompt": "rebook me on JX511"}
   1. ui_emitted       {"surface": "flights", "components": 17}
   2. user_action      {"action": "select_flight", "flight": "JX511"}
   3. ui_emitted       {"surface": "approve", "components": 7}
   4. user_action      {"action": "approve_booking", "context": {"flight": "JX511"}}
   5. policy_decision  {"action": "book_flight", "decision": "Allow", "reason": "customer within ceiling"}
   6. tool_executed    {"tool": "book_flight", "flight": "JX511"}
   7. ui_emitted       {"surface": "confirm", "components": 3}


## 6. TravelMind end-to-end on the AgentCore frame

One pass through every layer: Runtime receives the call, Identity supplies the role, the agent emits options, the user approves, Policy authorizes, the booking runs, and a confirmation surface streams back. Everything lands in the trace.

In [8]:
def travelmind_runtime(payload):
    """Full chain through the AgentCore-shaped frame. Returns the final surface + a trace."""
    trace, role = [], payload.get("role", "customer")
    claims = CLAIMS.get(role, CLAIMS["customer"])
    trace.append(("invoke", {"role": role, "prompt": payload.get("prompt", "")}))

    # 1. options
    opts = options_surface("flights", FLIGHTS, with_override=ROLE_CATALOG[role]["override"])
    if validate(opts):
        return {"final": "text fallback", "trace": trace}
    trace.append(("ui_emitted", {"surface": "flights"}))

    # 2. user selects + approves (simulated inbound userActions)
    target = FLIGHTS[1]
    trace.append(("user_action", {"action": "select_flight", "flight": target["id"]}))
    appr = approval_surface("approve", target); trace.append(("ui_emitted", {"surface": "approve"}))
    ua = user_action("approve_booking", "approve", "ok", {"flight": target["id"]})
    trace.append(("user_action", {"action": ua["userAction"]["name"]}))

    # 3. policy gate (server-side), then book
    amt = int(target["fare"].replace(",", ""))
    decision, why = evaluate_policy(claims, "book_flight", {"amount": amt})
    trace.append(("policy_decision", {"decision": decision, "reason": why}))
    if decision != "Allow":
        return {"final": f"blocked by policy: {why}", "trace": trace}
    conf = confirmation_surface("confirm", target)
    trace.append(("tool_executed", {"tool": "book_flight", "flight": target["id"]}))
    trace.append(("ui_emitted", {"surface": "confirm"}))
    return {"final": conf, "trace": trace}


result = travelmind_runtime({"prompt": "rebook me on the 13:40", "role": "customer"})
print("final surface streamed to the user:\n")
render_ascii(result["final"])
print("\ntrace:")
for i, (name, attrs) in enumerate(result["trace"]):
    print(f"  {i:>2}. {name:<16} {json.dumps(attrs)}")

final surface streamed to the user:

Column:
  +-- Booking confirmed: JX511 . INR 11,800 --+
  PNR JX48Q2 updated

trace:
   0. invoke           {"role": "customer", "prompt": "rebook me on the 13:40"}
   1. ui_emitted       {"surface": "flights"}
   2. user_action      {"action": "select_flight", "flight": "JX511"}
   3. ui_emitted       {"surface": "approve"}
   4. user_action      {"action": "approve_booking"}
   5. policy_decision  {"decision": "Allow", "reason": "customer within ceiling"}
   6. tool_executed    {"tool": "book_flight", "flight": "JX511"}
   7. ui_emitted       {"surface": "confirm"}


In [9]:
# deny path: an admin tries to book -> policy blocks it, no booking, clean trace
denied = travelmind_runtime({"prompt": "force the booking", "role": "admin"})
print("admin attempt ->", denied["final"])
print("trace tail:")
for name, attrs in denied["trace"][-2:]:
    print(f"  {name:<16} {json.dumps(attrs)}")

admin attempt -> blocked by policy: admin is read-only by policy
trace tail:
  user_action      {"action": "approve_booking"}
  policy_decision  {"decision": "Deny", "reason": "admin is read-only by policy"}


## 7. Dos and don'ts

**Do**

| Do | Why |
|---|---|
| Read the role from a verified Identity claim | The role decides which actions exist; do not infer it from the client |
| Enforce the gate in Policy (server-side) | A button click is a request, not an authorization |
| Emit one trace span per step | Observability must reproduce the full trajectory |
| Keep the entrypoint payload small and JSON-serializable | The Runtime contract is a JSON in, JSON out boundary |
| Set the region explicitly | The starter toolkit defaults to us-west-2; TravelMind is us-east-1 |

**Don't**

| Don't | Instead |
|---|---|
| Hide an action in the UI and call it security | Gate it in Policy; the surface is convenience |
| Trust a role passed in the prompt or context | Use the verified claim from Identity |
| Skip the deny path in testing | Prove the gate blocks, not just that the happy path works |
| Block on a live deploy to teach the pattern | Run the entrypoint locally; deploy is a separate step |
| Treat preview features as stable | Policy is in preview; pin versions and plan for change |

The same TravelMind anchor carried both notebooks. Notebook 1 taught the A2UI pattern; this one placed it in the AWS production frame.